# Generative AI — Assignment 2: Smart Mutual Fund Advisor

**Note:** No starter notebook was uploaded for this assignment, so this notebook is written from the assignment brief, using the same LangChain / Gradio / Groq / ChromaDB stack as your earlier assignments. Before submitting:
- Update `CSV_PATH` to your actual Mutual Fund dataset file and check the column names match.
- Confirm the model names in `MODEL_MAP` and the embedding model match what was taught in class.
- Run the notebook top to bottom with a valid `GROQ_API_KEY` and Ollama running locally.

In [ ]:
# pip install langchain langchain-community langchain-groq langchain-ollama langchain-chroma \
#     langgraph gradio pydantic python-dotenv pandas

In [ ]:
import os
import uuid
import pandas as pd
from dotenv import load_dotenv

from langchain_community.document_loaders import CSVLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_groq import ChatGroq
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from pydantic import BaseModel, Field
from typing import List
from typing_extensions import TypedDict

import gradio as gr
from langgraph.graph import StateGraph, START, END

## Set environment variables and API Keys (2.5 marks)

In [ ]:
## TODO: Load environment variables and API Keys here
load_dotenv()
GROQ_API_KEY = os.environ["GROQ_API_KEY"]

## Define Chat Model (2.5 marks)

In [ ]:
## TODO: Model name map for UI Mapping
MODEL_MAP = {
    "GPT OSS 120B": "openai/gpt-oss-120b",
    ## TODO: Add 2 more models here
    "Llama 3.3 70B": "llama-3.3-70b-versatile",
    ## TODO: Add 2 more models here
    "Llama 3.1 8B Instant": "llama-3.1-8b-instant",
}


## TODO: Model Selector
def get_llm(model_label: str, temperature: float):
    return ChatGroq(model=MODEL_MAP[model_label], temperature=temperature, api_key=GROQ_API_KEY)

## Define Embedding Model (2.5 marks)

In [ ]:
## TODO: Create embeddings
embeddings = OllamaEmbeddings(model="llama3.2")

## Define Vector Store (2.5 marks)

In [ ]:
## TODO: Initialize ChromaDB with a persist directory set
vector_store = Chroma(
    collection_name="mutual_funds",
    embedding_function=embeddings,
    persist_directory="./chroma_db_mutual_funds",
)

## Load and Preprocess Data (5 marks)

In [ ]:
CSV_PATH = "mutual_funds_data.csv"  # update to your dataset path

## TODO: Keep Scheme Name, Fund House, Scheme Type, Scheme Category, Net Asset Value and Date in content
## TODO: Keep Scheme Type, Scheme Category in metadata
df = pd.read_csv(CSV_PATH)

documents = []
for _, row in df.iterrows():
    content = (
        f"Scheme Name: {row['Scheme Name']}\n"
        f"Fund House: {row['Fund House']}\n"
        f"Scheme Type: {row['Scheme Type']}\n"
        f"Scheme Category: {row['Scheme Category']}\n"
        f"Net Asset Value: {row['Net Asset Value']}\n"
        f"Date: {row['Date']}"
    )
    metadata = {
        "Scheme Type": row["Scheme Type"],
        "Scheme Category": row["Scheme Category"],
    }
    documents.append(Document(page_content=content, metadata=metadata))

len(documents)

## Store in Vector DB (5 marks)

In [ ]:
## TODO: Add UUIDs
uuids = [str(uuid.uuid4()) for _ in documents]

## TODO: Add documents
vector_store.add_documents(documents=documents, ids=uuids)

## Initialize Retriever (5 marks)

In [ ]:
## TODO: Initializer Retriever below
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

## Define Pydantic Schema (10 marks)

In [ ]:
## TODO: Define Pydantic schema with type and description
class Fund(BaseModel):
    scheme_name: str = Field(description="Name of the recommended mutual fund scheme")
    fund_house: str = Field(description="Fund house / AMC managing the scheme")


class FundRecommendation(BaseModel):
    suggestion: str = Field(description="Natural language explanation of what is recommended and why")
    recommended_funds: List[Fund] = Field(description="List of recommended mutual fund schemes")


output_parser = PydanticOutputParser(pydantic_object=FundRecommendation)

## Define Prompt (5 marks)

In [ ]:
## TODO: Use ChatPromptTemplate.from_template with fields: preferences, context, format_instructions
prompt = ChatPromptTemplate.from_template(
    """You are a mutual fund investment advisor. Based on the user's preferences and the
retrieved mutual fund data below, recommend suitable schemes.

User Preferences: {preferences}

Relevant Mutual Funds:
{context}

{format_instructions}
"""
)

## Test RAG Pipeline Manually (10 marks)

In [ ]:
preferences = "high return portfolio, only from Aditya Birla fund house"

## TODO: Retrieve relevant funds
relevant_docs = retriever.invoke(preferences)

## TODO: Joining the funds
context = "\n\n".join(doc.page_content for doc in relevant_docs)

## TODO: Get LLM
llm = get_llm("Llama 3.3 70B", 0.7)

## TODO: Create a Runnable Sequence with 3 Runnables - template, LLM and Output Parser (2.5 marks)
test_chain = prompt | llm | output_parser

## TODO: Create dictionary with preferences, context and format_instructions
test_input = {
    "preferences": preferences,
    "context": context,
    "format_instructions": output_parser.get_format_instructions(),
}

## TODO: Invoke chain and the output of this command should be a dictionary (2.5 marks)
test_result = test_chain.invoke(test_input).model_dump()
test_result

## Define RAG Pipeline Function (10 marks)

In [ ]:
def rag_pipeline(preferences: str, model_label: str, temperature: float) -> str:
    ## TODO: Retrieving relevant funds
    relevant_docs = retriever.invoke(preferences)

    ## TODO: Joining the funds
    context = "\n\n".join(doc.page_content for doc in relevant_docs)

    ## TODO: Get the LLM
    llm = get_llm(model_label, temperature)

    ## TODO: Create a Runnable Sequence with 2 Runnables - template and LLM (2.5 marks)
    chain = prompt | llm

    ## TODO: Invoke the Runnable Sequence with the preferences, context and format instructions (2.5 marks)
    response = chain.invoke({
        "preferences": preferences,
        "context": context,
        "format_instructions": output_parser.get_format_instructions(),
    })
    return response.content

## Gradio App

In [ ]:
def handle_submit(preferences, model_label, temperature):
    raw_output = rag_pipeline(preferences, model_label, temperature)
    try:
        parsed = output_parser.parse(raw_output)
    except Exception:
        parsed = FundRecommendation(suggestion=raw_output, recommended_funds=[])
    funds_table = [[f.scheme_name, f.fund_house] for f in parsed.recommended_funds]
    return parsed.suggestion, funds_table


with gr.Blocks() as demo:
    gr.Markdown("# Smart Mutual Fund Advisor")
    preferences_input = gr.Textbox(
        label="Investment Needs",
        placeholder="e.g. high return portfolio, only from Aditya Birla fund house",
    )
    with gr.Accordion("LLM Settings (Advanced)", open=False):
        model_dropdown = gr.Dropdown(choices=list(MODEL_MAP.keys()), value="GPT OSS 120B", label="Model")
        temperature_slider = gr.Slider(0, 2, value=0.7, step=0.1, label="Temperature")
    submit_btn = gr.Button("Get Recommendation")
    suggestion_output = gr.Textbox(label="Suggestion", interactive=False)
    funds_output = gr.Dataframe(headers=["Scheme Name", "Fund House"], label="Recommended Mutual Funds")

    submit_btn.click(
        handle_submit,
        inputs=[preferences_input, model_dropdown, temperature_slider],
        outputs=[suggestion_output, funds_output],
    )

demo.launch()

# Part 2: RAG Pipeline in LangGraph

## Define State Class (7.5 marks)

In [ ]:
## TODO: Define State class with preferences, context and answer keys
class RAGState(TypedDict):
    preferences: str
    context: str
    answer: str

## Define Retrieve Node (7.5 marks)

In [ ]:
## TODO: Define retrieve node
def retrieve(state: RAGState):
    ## TODO: Retrieving relevant mutual funds
    docs = retriever.invoke(state["preferences"])
    context = "\n\n".join(doc.page_content for doc in docs)
    ## TODO: Return the relevant parts of the state: context only
    return {"context": context}

## Define Generate Node (10 marks)

In [ ]:
## TODO: Define generate node
def generate(state: RAGState):
    ## TODO: Joining the mutual funds
    # (context is already joined by the retrieve node)

    ## TODO: Create a Runnable Sequence with 2 Runnables - template and llm_graph (different variable that's defined later)
    chain = prompt | llm_graph

    ## TODO: Invoke the Runnable Sequence with the preferences, context and format instructions
    response = chain.invoke({
        "preferences": state["preferences"],
        "context": state["context"],
        "format_instructions": output_parser.get_format_instructions(),
    })

    ## TODO: Return the relevant parts of the state: answer only
    return {"answer": response.content}

## Build and Compile Graph (10 marks)

In [ ]:
## TODO: Build the graph
graph_builder = StateGraph(RAGState)
## TODO: To add
graph_builder.add_node("retrieve", retrieve)
## TODO: To add
graph_builder.add_node("generate", generate)
## TODO: To add
graph_builder.add_edge(START, "retrieve")
## TODO: To add
graph_builder.add_edge("retrieve", "generate")
## TODO: To add
graph_builder.add_edge("generate", END)

graph = graph_builder.compile()

## Test Manually (5 marks)

In [ ]:
## TODO: Get LLM
llm_graph = get_llm("Llama 3.3 70B", 0.7)

## TODO: Invoke the graph
graph_result = graph.invoke({
    "preferences": "high return portfolio, only from Aditya Birla fund house",
    "context": "",
    "answer": "",
})
graph_result["answer"]